# Metric- and Architecture-Dependent Fairness Effects of Differential Privacy in Educational Knowledge Tracing

**Submission-ready final v2 reproducibility and paper-audit notebook**

This notebook is designed for two complementary uses:

1. **Paper writing / review artifact** (`EXECUTION_PROFILE = "tables_only"`): regenerate the final paper-ready tables and figures from the completed D8–D16 experimental snapshots.
2. **Full reproduction** (`EXECUTION_PROFILE = "full"`): rebuild the OULAD sequence pipeline and rerun the training/evaluation suites from raw CSVs.

The experimental arc is:

- **D8/D9**: identify DP-SGD-induced utility loss and IMD-based ranking fairness risk.
- **D10**: diagnose group-wise loss, gradient norms, and clipping-rate differences.
- **D11–D14**: show that ordinary weighting and simple ranking regularization provide limited mitigation.
- **D12**: test SAKT architecture robustness.
- **D15/D16**: evaluate and confirm LIRE/LIRE-WGPR as the best DKT-specific mitigation candidate.

The notebook intentionally separates **measurement claims**, **mechanism claims**, and **mitigation claims**. It avoids overclaiming: LIRE is reported as a promising DKT-specific Pareto candidate, not as a universal solution to DP fairness risk in KT.


## Literature and method map

| Component | Role in this notebook | Citation anchor |
|---|---|---|
| OULAD | Main dataset; contains demographic information including `imd_band` for socioeconomic grouping | Kuzilek, Hlosta, and Zdrahal, *Scientific Data*, 2017 |
| DKT | Recurrent KT backbone | Piech et al., *NeurIPS*, 2015 |
| SAKT | Self-attentive KT robustness model | Pandey and Karypis, *EDM*, 2019 |
| AKT | Related attention-based KT model; discussed but not required in the final pipeline | Ghosh, Heffernan, and Lan, *KDD*, 2020 |
| DP-SGD / Opacus | Private training, clipping, noise, and privacy accounting | Abadi et al., 2016; Opacus documentation |
| DP and group disparity | Motivation for studying disparate impact under DP | Bagdasaryan & Shmatikov, 2019; Mangold et al., 2023; Hansen et al., 2024 |
| Pairwise AUC surrogate | Motivation for ranking-aware mitigation | AUC optimization / pairwise ranking literature |
| Clipping-aware mitigation | Motivation for clipping-threshold sweep and LIRE diagnostics | recent clipping-bias / adaptive clipping / error-feedback DP literature |
| LIRE | This work: Low-IMD Residual Expert for DKT-specific mitigation | proposed in this notebook |

**Important scope note.** ASSISTments is a valuable KT benchmark, but the public ASSISTments 2009 fields do not provide an OULAD-like socioeconomic attribute comparable to `imd_band`. Therefore, this final notebook keeps fairness evidence on OULAD-IMD and treats external KT datasets as optional privacy–utility appendices rather than socioeconomic fairness evidence.


## Execution profiles and reproducibility

Full reproduction can take many hours on Kaggle free GPUs. The recommended workflow is:

- `EXECUTION_PROFILE = "tables_only"`: generate final paper-ready summary tables and figures from embedded completed results. This is suitable for manuscript writing and internal review.
- `EXECUTION_PROFILE = "core"`: run the core DKT DP experiments and metrics.
- `EXECUTION_PROFILE = "full"`: run the complete D8–D16 experimental program from raw OULAD CSVs.

To reproduce from raw OULAD CSVs, add the OULAD dataset to the Kaggle notebook and set:

```python
EXECUTION_PROFILE = "full"
```

The embedded final tables are not a substitute for raw reproducibility; they are included so that the paper figures and result tables can be regenerated quickly and deterministically during writing.


In [ ]:
# ============================================================
# 0. Environment and dependency setup
# ============================================================

import os
import sys
import subprocess
import importlib
from pathlib import Path

EXECUTION_PROFILE = os.getenv("EXECUTION_PROFILE", "tables_only")  # tables_only, core, full
INSTALL_MISSING_PACKAGES = os.getenv("INSTALL_MISSING_PACKAGES", "0") == "1"


def ensure_package(import_name, pip_name=None):
    pip_name = pip_name or import_name
    try:
        importlib.import_module(import_name)
        print(f"{import_name} is already available.")
    except ImportError:
        if not INSTALL_MISSING_PACKAGES:
            raise
        print(f"Installing {pip_name}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])
        print(f"Installed {pip_name}.")

ensure_package("opacus", "opacus")

In [ ]:
# ============================================================
# 1. Imports and global configuration
# ============================================================

import os
import gc
import json
import math
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import roc_auc_score, brier_score_loss
from IPython.display import display

from opacus import PrivacyEngine
from opacus.validators import ModuleValidator
try:
    from opacus.layers import DPLSTM
except ImportError:
    from opacus.layers.dp_lstm import DPLSTM

# Opacus does not support torch.nn.MultiheadAttention directly.
# For SAKT full-mode DP training, this notebook uses a small custom
# batch-first self-attention block built from nn.Linear + tensor ops.
# This avoids DPMultiheadAttention's sequence-first grad_sample/mask edge cases.

warnings.filterwarnings("ignore")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

ROOT_DIR = Path(os.getenv("OUTPUT_DIR", Path.cwd() / "artifacts")).expanduser().resolve()
DATA_CACHE_DIR = ROOT_DIR / "cache"
PRED_DIR = ROOT_DIR / "predictions"
RESULT_DIR = ROOT_DIR / "results"
FIG_DIR = ROOT_DIR / "figures"
for d in [ROOT_DIR, DATA_CACHE_DIR, PRED_DIR, RESULT_DIR, FIG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

CONFIG = {
    "SEEDS": [42, 43, 44],
    "SPLIT_SEED": 2026,
    "MAX_LEN": 100,
    "BATCH_SIZE_DKT": 1024,
    "BATCH_SIZE_SAKT": 512,
    "HIDDEN_DIM": 64,
    "SAKT_D_MODEL": 64,
    "SAKT_N_HEADS": 4,
    "EPOCHS": 8,
    "LR": 1e-3,
    "MAX_GRAD_NORM": 1.0,
    "TARGET_DELTA": 1e-5,
    "NUM_WORKERS": 0,
    "MAIN_EPSILONS": [10.0, 1.0, 0.5],
    "CONFIRM_EPSILON": 1.0,
    "ROBUST_REPEATS": 300,
    "PRIMARY_ATTR": "imd_low30_vs_rest",
    "FOCUS_ATTRS": ["imd_low30_vs_rest", "imd_low30_vs_high70"],
}

def make_config_signature(config, model="dkt"):
    bs = config["BATCH_SIZE_DKT"] if model == "dkt" else config["BATCH_SIZE_SAKT"]
    dim = config["HIDDEN_DIM"] if model == "dkt" else config["SAKT_D_MODEL"]
    return (
        f"{model}_bs{bs}_len{config['MAX_LEN']}_d{dim}"
        f"_ep{config['EPOCHS']}_lr{str(config['LR']).replace('.', 'p')}"
        f"_C{str(config['MAX_GRAD_NORM']).replace('.', 'p')}_split{config['SPLIT_SEED']}"
    )

CONFIG_SIGNATURE_DKT = make_config_signature(CONFIG, "dkt")
CONFIG_SIGNATURE_SAKT = make_config_signature(CONFIG, "sakt")
print("DKT signature:", CONFIG_SIGNATURE_DKT)
print("SAKT signature:", CONFIG_SIGNATURE_SAKT)

with open(ROOT_DIR / "submission_config.json", "w") as f:
    json.dump(CONFIG, f, indent=2)

In [ ]:
# ============================================================
# 2. Reproducibility helpers
# ============================================================

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

try:
    torch.set_num_threads(2)
    torch.set_num_interop_threads(1)
except RuntimeError:
    pass


def safe_get_epsilon(privacy_engine, delta):
    try:
        eps = privacy_engine.get_epsilon(delta)
        if eps is None or not np.isfinite(eps):
            return np.nan
        return float(eps)
    except Exception as e:
        print(f"[Warning] get_epsilon failed: {type(e).__name__}: {e}")
        return np.nan


def method_tag(method_name):
    return (
        method_name
        .replace(" ", "_")
        .replace("=", "")
        .replace(".", "p")
        .replace("/", "_")
    )


def epsilon_tag(epsilon):
    if epsilon is None:
        return "NoDP"
    return f"eps{str(epsilon).replace('.', 'p')}"

# Data: OULAD sequence construction and IMD grouping

The Open University Learning Analytics Dataset includes student demographics, assessments, and virtual learning environment interactions. This notebook uses the assessment sequence as the KT interaction stream and uses `imd_band` as the socioeconomic grouping variable.

Two IMD contrasts are used:

- `imd_low30_vs_rest`: group 1 = IMD 0–30%, group 0 = IMD 30–100%. This is the primary analysis because it compares the most socioeconomically disadvantaged group against the rest of the student population.
- `imd_low30_vs_high70`: group 1 = IMD 0–30%, group 0 = IMD 70–100%. This is a cleaner robustness contrast because it compares low-IMD and high-IMD students more directly and reduces group-size concerns.

In [ ]:
# ============================================================
# 3. OULAD data loading and KT sequence construction
# ============================================================

import pickle


def find_oulad_base_path():
    configured = os.getenv("OULAD_DIR")
    candidates = [
        Path(configured).expanduser() if configured else None,
        Path.cwd() / "data" / "oulad",
        Path.cwd() / "data",
        Path("/kaggle/input/open-university-learning-analytics-dataset"),
        Path("/kaggle/input/oulad"),
        Path("/kaggle/input/open-university-learning-analytics-dataset-anonymised"),
    ]
    for path in candidates:
        if path is not None and (path / "studentInfo.csv").exists():
            return path.resolve()

    kaggle_root = Path("/kaggle/input")
    if kaggle_root.exists():
        for csv_path in kaggle_root.rglob("studentInfo.csv"):
            return csv_path.parent.resolve()

    raise FileNotFoundError(
        "OULAD files were not found. Set OULAD_DIR to the directory containing "
        "studentInfo.csv, studentAssessment.csv, and assessments.csv."
    )


def map_imd_low30_vs_rest(v):
    if pd.isna(v) or v == "?":
        return -1
    low30 = {"0-10%", "10-20", "10-20%", "20-30", "20-30%"}
    return 1 if v in low30 else 0


def map_imd_low30_vs_high70(v):
    if pd.isna(v) or v == "?":
        return -1
    low30 = {"0-10%", "10-20", "10-20%", "20-30", "20-30%"}
    high70 = {"70-80", "70-80%", "80-90", "80-90%", "90-100", "90-100%"}
    if v in low30:
        return 1
    if v in high70:
        return 0
    return -1


def build_oulad_sequences(base_path):
    student_info = pd.read_csv(base_path / "studentInfo.csv")
    student_assessment = pd.read_csv(base_path / "studentAssessment.csv")
    assessments = pd.read_csv(base_path / "assessments.csv")

    if "assessment_id" in student_assessment.columns and "id_assessment" not in student_assessment.columns:
        student_assessment = student_assessment.rename(columns={"assessment_id": "id_assessment"})
    if "assessment_id" in assessments.columns and "id_assessment" not in assessments.columns:
        assessments = assessments.rename(columns={"assessment_id": "id_assessment"})

    if "is_banked" in student_assessment.columns:
        student_assessment = student_assessment[student_assessment["is_banked"] == 0].copy()

    data = student_assessment.merge(assessments, on="id_assessment", how="left")
    data = data.merge(student_info, on=["id_student", "code_module", "code_presentation"], how="left")

    data = data.dropna(subset=["score", "date_submitted"])
    data = data[data["score"] >= 0].copy()
    data["correct"] = (data["score"] >= 50).astype(int)

    unique_questions = sorted(data["id_assessment"].dropna().unique())
    q2id = {q: i for i, q in enumerate(unique_questions)}
    data["q_id"] = data["id_assessment"].map(q2id).astype(int)

    data["student_key"] = (
        data["id_student"].astype(str) + "_" +
        data["code_module"].astype(str) + "_" +
        data["code_presentation"].astype(str)
    )

    data["group_imd_low30_vs_rest"] = data["imd_band"].apply(map_imd_low30_vs_rest)
    data["group_imd_low30_vs_high70"] = data["imd_band"].apply(map_imd_low30_vs_high70)
    data = data.sort_values(["student_key", "date_submitted", "id_assessment"])

    sequences = []
    for key, gdf in data.groupby("student_key"):
        seq = list(zip(gdf["q_id"].astype(int), gdf["correct"].astype(int)))
        if len(seq) < 2:
            continue
        first = gdf.iloc[0]
        sequences.append({
            "student_key": key,
            "sequence": seq,
            "attrs": {
                "imd_band": first.get("imd_band", np.nan),
                "group_imd_low30_vs_rest": int(first["group_imd_low30_vs_rest"]),
                "group_imd_low30_vs_high70": int(first["group_imd_low30_vs_high70"]),
            }
        })
    return sequences, q2id


def load_or_build_sequences(force_rebuild=False):
    cache_path = DATA_CACHE_DIR / f"oulad_sequences_len{CONFIG['MAX_LEN']}_split{CONFIG['SPLIT_SEED']}.pkl"
    if cache_path.exists() and not force_rebuild:
        with open(cache_path, "rb") as f:
            cache = pickle.load(f)
        print("Loaded sequence cache:", cache_path)
        return cache["sequences"], cache["q2id"]

    base_path = find_oulad_base_path()
    print("OULAD base path:", base_path)
    sequences, q2id = build_oulad_sequences(base_path)
    with open(cache_path, "wb") as f:
        pickle.dump({"sequences": sequences, "q2id": q2id}, f)
    print("Saved sequence cache:", cache_path)
    return sequences, q2id


def split_sequences(sequences, seed=2026, train_ratio=0.70, val_ratio=0.15):
    set_seed(seed)
    seqs = list(sequences)
    random.shuffle(seqs)
    n = len(seqs)
    train = seqs[:int(train_ratio * n)]
    val = seqs[int(train_ratio * n):int((train_ratio + val_ratio) * n)]
    test = seqs[int((train_ratio + val_ratio) * n):]
    return train, val, test


def summarize_imd_groups(sequences):
    rows = []
    for attr in CONFIG["FOCUS_ATTRS"]:
        col = f"group_{attr}"
        vals = np.array([s["attrs"][col] for s in sequences])
        rows.append({
            "attribute": attr,
            "group0_sequences": int((vals == 0).sum()),
            "group1_sequences": int((vals == 1).sum()),
            "missing_or_excluded": int((vals == -1).sum()),
            "total_sequences": len(vals),
        })
    return pd.DataFrame(rows)

if EXECUTION_PROFILE in ["core", "full"]:
    sequences, q2id = load_or_build_sequences()
    Q = len(q2id)
    train_data, val_data, test_data = split_sequences(sequences, CONFIG["SPLIT_SEED"])
    print("Q:", Q, "Train/Val/Test:", len(train_data), len(val_data), len(test_data))
    display(summarize_imd_groups(sequences))
else:
    print("Skipping raw data construction because EXECUTION_PROFILE='tables_only'.")
    sequences, q2id, Q, train_data, val_data, test_data = None, None, None, None, None, None

In [ ]:
# ============================================================
# 4. Dataset and DataLoader
# ============================================================

class KTDataset(Dataset):
    def __init__(self, data, q_num, max_len=100):
        self.data = data
        self.q_num = q_num
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        seq = item["sequence"]
        q_seq = [x[0] for x in seq]
        y_seq = [x[1] for x in seq]

        x_input, q_target, y_target = [], [], []
        for i in range(len(q_seq) - 1):
            q_t, y_t = q_seq[i], y_seq[i]
            q_next, y_next = q_seq[i + 1], y_seq[i + 1]
            x_input.append(q_t + y_t * self.q_num)
            q_target.append(q_next)
            y_target.append(y_next)

        if len(x_input) < self.max_len:
            pad = self.max_len - len(x_input)
            x_input = [0] * pad + x_input
            q_target = [0] * pad + q_target
            y_target = [-1] * pad + y_target
        else:
            x_input = x_input[-self.max_len:]
            q_target = q_target[-self.max_len:]
            y_target = y_target[-self.max_len:]

        attrs = item["attrs"]
        return (
            torch.tensor(x_input, dtype=torch.long),
            torch.tensor(q_target, dtype=torch.long),
            torch.tensor(y_target, dtype=torch.float),
            torch.tensor(attrs["group_imd_low30_vs_rest"], dtype=torch.long),
            torch.tensor(attrs["group_imd_low30_vs_high70"], dtype=torch.long),
        )


def build_dataloaders(q_num, batch_size, model_type="dkt"):
    train_dataset = KTDataset(train_data, q_num, CONFIG["MAX_LEN"])
    val_dataset = KTDataset(val_data, q_num, CONFIG["MAX_LEN"])
    test_dataset = KTDataset(test_data, q_num, CONFIG["MAX_LEN"])

    train_loader = DataLoader(
        train_dataset, batch_size=batch_size, shuffle=True, drop_last=False,
        num_workers=CONFIG["NUM_WORKERS"], pin_memory=True
    )
    val_loader = DataLoader(
        val_dataset, batch_size=batch_size, shuffle=False, drop_last=False,
        num_workers=CONFIG["NUM_WORKERS"], pin_memory=True
    )
    test_loader = DataLoader(
        test_dataset, batch_size=batch_size, shuffle=False, drop_last=False,
        num_workers=CONFIG["NUM_WORKERS"], pin_memory=True
    )
    return train_dataset, val_dataset, test_dataset, train_loader, val_loader, test_loader

# Models

Two KT architectures are implemented.

1. **DKT**: embedding + recurrent state + question-specific output. For DP runs, `DPLSTM` is used to remain compatible with Opacus.
2. **SAKT-style self-attention**: causal self-attention over historical interactions with a query embedding for the target item.

The final mitigation uses **LIRE**: a Low-IMD Residual Expert. LIRE adds a small group-adaptive residual output correction only for low-IMD students. The key design is not to enlarge the low-IMD loss weight directly, but to provide limited additional output capacity to the disadvantaged group. In the code, the residual expert is only instantiated for LIRE methods to avoid unused-parameter errors in Opacus.

### Full-mode Opacus compatibility note

SAKT is implemented with `opacus.layers.DPMultiheadAttention`, not `torch.nn.MultiheadAttention`. Opacus cannot compute per-sample gradients for the fused PyTorch implementation of `nn.MultiheadAttention`, so the DP-compatible replacement is required for `EXECUTION_PROFILE = "full"`.

**Implementation note.** The SAKT attention block uses `DPMultiheadAttention(batch_first=False)` internally. The surrounding model remains batch-first, but query/key/value tensors are transposed to `[seq_len, batch_size, dim]` only for the attention call. This avoids Opacus attn-mask validation errors in full DP training.


### Full-mode SAKT DP compatibility note

For full reproduction, SAKT uses a custom batch-first self-attention block instead of `torch.nn.MultiheadAttention`. This is necessary because Opacus does not support the standard PyTorch multi-head attention module directly, and some `DPMultiheadAttention` versions can expose sequence-length-leading `grad_sample` tensors in sequence-first mode. The custom block keeps all trainable projections as batch-first `nn.Linear` layers, preserving Opacus per-sample gradient alignment while implementing the same causal self-attention structure needed by SAKT.

In [ ]:
# ============================================================
# 5. Model definitions: DKT, DKT-LIRE, SAKT, SAKT-LIRE
# Note: SAKT avoids torch.nn.MultiheadAttention because Opacus does not
# support it directly. We use a lightweight batch-first self-attention
# block made of nn.Linear layers and tensor operations, which keeps
# per-sample gradients aligned with the batch dimension.
# ============================================================

class DKTModel(nn.Module):
    def __init__(self, q_num, hidden_dim=64, use_dp_lstm=True, use_lire=False, lire_alpha=0.0):
        super().__init__()
        self.q_num = q_num
        self.hidden_dim = hidden_dim
        self.use_lire = bool(use_lire)
        self.lire_alpha = float(lire_alpha)
        self.embedding = nn.Embedding(2 * q_num, hidden_dim)
        if use_dp_lstm:
            self.rnn = DPLSTM(hidden_dim, hidden_dim, batch_first=True)
        else:
            self.rnn = nn.LSTM(hidden_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, q_num)
        self.residual_fc = nn.Linear(hidden_dim, q_num) if self.use_lire else None

    def forward(self, x, q_target, g_rest=None):
        emb = self.embedding(x)
        h, _ = self.rnn(emb)
        logits_all = self.fc(h)
        selected = torch.gather(logits_all, dim=2, index=q_target.unsqueeze(-1)).squeeze(-1)

        if self.use_lire:
            residual_all = self.residual_fc(h)
            residual = torch.gather(residual_all, dim=2, index=q_target.unsqueeze(-1)).squeeze(-1)
            if g_rest is None:
                group_gate = torch.ones_like(selected)
            else:
                group_gate = (g_rest.unsqueeze(1).expand_as(selected) == 1).float()
            selected = selected + self.lire_alpha * group_gate * residual
        return selected



class BatchFirstDPSelfAttention(nn.Module):
    """A minimal multi-head self-attention block compatible with Opacus.

    Why not nn.MultiheadAttention?
    Opacus does not support torch.nn.MultiheadAttention directly. DPMultiheadAttention
    is available, but in some Kaggle/Opacus versions its sequence-first internals can
    produce grad_sample tensors whose leading dimension is seq_len rather than batch_size,
    which breaks both diagnostics and sometimes clipping. This implementation keeps all
    trainable projections batch-first through nn.Linear, so per-sample gradients keep
    the expected first dimension: batch_size.
    """
    def __init__(self, d_model, n_heads, dropout=0.0):
        super().__init__()
        if d_model % n_heads != 0:
            raise ValueError("d_model must be divisible by n_heads")
        self.d_model = d_model
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def _split_heads(self, z):
        # z: [B, L, D] -> [B, H, L, Dh]
        bsz, seq_len, _ = z.shape
        return z.view(bsz, seq_len, self.n_heads, self.head_dim).transpose(1, 2).contiguous()

    def forward(self, query, key, value, attn_mask=None):
        q = self._split_heads(self.q_proj(query))
        k = self._split_heads(self.k_proj(key))
        v = self._split_heads(self.v_proj(value))

        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        if attn_mask is not None:
            # attn_mask: [L, L] with True/1 indicating positions to mask.
            if attn_mask.dtype != torch.bool:
                attn_mask = attn_mask.bool()
            scores = scores.masked_fill(attn_mask.unsqueeze(0).unsqueeze(0), -1e4)
        weights = torch.softmax(scores, dim=-1)
        weights = self.dropout(weights)
        out = torch.matmul(weights, v)  # [B, H, L, Dh]
        out = out.transpose(1, 2).contiguous().view(query.shape[0], query.shape[1], self.d_model)
        return self.out_proj(out)


class SAKTModel(nn.Module):
    def __init__(self, q_num, d_model=64, n_heads=4, max_len=100, use_lire=False, lire_alpha=0.0):
        super().__init__()
        self.q_num = q_num
        self.d_model = d_model
        self.max_len = max_len
        self.use_lire = bool(use_lire)
        self.lire_alpha = float(lire_alpha)

        self.interaction_emb = nn.Embedding(2 * q_num, d_model)
        self.question_emb = nn.Embedding(q_num, d_model)
        self.pos_emb = nn.Embedding(max_len, d_model)
        self.attn = BatchFirstDPSelfAttention(d_model, n_heads, dropout=0.0)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.ReLU(),
            nn.Linear(d_model, d_model),
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.out = nn.Linear(d_model, 1)
        self.residual_out = nn.Linear(d_model, 1) if self.use_lire else None

    def forward(self, x, q_target, g_rest=None):
        bsz, seq_len = x.shape
        pos = torch.arange(seq_len, device=x.device).unsqueeze(0).expand(bsz, seq_len)
        k_v = self.interaction_emb(x) + self.pos_emb(pos)
        q = self.question_emb(q_target) + self.pos_emb(pos)
        attn_mask = torch.triu(torch.ones(seq_len, seq_len, device=x.device), diagonal=1).bool()
        attn_out = self.attn(q, k_v, k_v, attn_mask=attn_mask)
        h = self.norm1(q + attn_out)
        h = self.norm2(h + self.ffn(h))
        logits = self.out(h).squeeze(-1)
        if self.use_lire:
            residual = self.residual_out(h).squeeze(-1)
            if g_rest is None:
                group_gate = torch.ones_like(logits)
            else:
                group_gate = (g_rest.unsqueeze(1).expand_as(logits) == 1).float()
            logits = logits + self.lire_alpha * group_gate * residual
        return logits

def build_model(q_num, model_type, method_spec):
    use_lire = method_spec.get("use_lire", False)
    lire_alpha = method_spec.get("lire_alpha", 0.0)
    if model_type == "dkt":
        return DKTModel(
            q_num=q_num,
            hidden_dim=CONFIG["HIDDEN_DIM"],
            use_dp_lstm=method_spec.get("use_dp", True),
            use_lire=use_lire,
            lire_alpha=lire_alpha,
        )
    if model_type == "sakt":
        return SAKTModel(
            q_num=q_num,
            d_model=CONFIG["SAKT_D_MODEL"],
            n_heads=CONFIG["SAKT_N_HEADS"],
            max_len=CONFIG["MAX_LEN"],
            use_lire=use_lire,
            lire_alpha=lire_alpha,
        )
    raise ValueError(f"Unknown model_type: {model_type}")

In [ ]:
# ============================================================
# 6. Losses: BCE, static weighting, PairRank, warm-up PairRank
# ============================================================

criterion_none = nn.BCEWithLogitsLoss(reduction="none")


def group_weight_matrix(y, g_rest, method_spec):
    g0_w = float(method_spec.get("group0_weight", 1.0))
    g1_w = float(method_spec.get("group1_weight", 1.0))
    group_token = g_rest.unsqueeze(1).expand_as(y)
    w = torch.ones_like(y, dtype=torch.float)
    w = torch.where(group_token == 0, torch.full_like(w, g0_w), w)
    w = torch.where(group_token == 1, torch.full_like(w, g1_w), w)
    return w


def within_student_pairrank_loss(logits, y, mask, g_rest=None, target_group=1, tau=0.2, max_pairs_per_student=128):
    losses = []
    bsz = logits.shape[0]
    for i in range(bsz):
        if g_rest is not None and int(g_rest[i].detach().cpu().item()) != target_group:
            continue
        valid = mask[i]
        if valid.sum() < 2:
            continue
        scores = logits[i][valid]
        labels = y[i][valid]
        pos = scores[labels == 1]
        neg = scores[labels == 0]
        if len(pos) == 0 or len(neg) == 0:
            continue
        if len(pos) * len(neg) > max_pairs_per_student:
            p_idx = torch.randint(0, len(pos), (max_pairs_per_student,), device=logits.device)
            n_idx = torch.randint(0, len(neg), (max_pairs_per_student,), device=logits.device)
            diffs = pos[p_idx] - neg[n_idx]
        else:
            diffs = pos.unsqueeze(1) - neg.unsqueeze(0)
        losses.append(torch.nn.functional.softplus(-diffs / tau).mean())
    if not losses:
        return logits.sum() * 0.0
    return torch.stack(losses).mean()


def effective_rank_lambda(method_spec, epoch_idx):
    lam = float(method_spec.get("rank_lambda", 0.0))
    warmup = int(method_spec.get("rank_warmup_epochs", 0))
    if epoch_idx < warmup:
        return 0.0
    return lam


def compute_training_loss(logits, y, g_rest, method_spec, epoch_idx):
    mask = y != -1
    if mask.sum() == 0:
        return logits.sum() * 0.0, {"bce_loss": np.nan, "rank_loss": 0.0, "lambda_eff": 0.0}

    raw = criterion_none(logits, torch.clamp(y, min=0.0))
    weights = group_weight_matrix(y, g_rest, method_spec)
    bce = (raw[mask] * weights[mask]).sum() / weights[mask].sum().clamp_min(1.0)

    lam_eff = effective_rank_lambda(method_spec, epoch_idx)
    rank_loss = logits.sum() * 0.0
    if lam_eff > 0:
        rank_loss = within_student_pairrank_loss(
            logits=logits,
            y=y,
            mask=mask,
            g_rest=g_rest,
            target_group=1,
            tau=float(method_spec.get("rank_tau", 0.2)),
            max_pairs_per_student=int(method_spec.get("max_pairs_per_student", 128)),
        )
    total = bce + lam_eff * rank_loss
    return total, {
        "bce_loss": float(bce.detach().cpu().item()),
        "rank_loss": float(rank_loss.detach().cpu().item()) if torch.is_tensor(rank_loss) else float(rank_loss),
        "lambda_eff": lam_eff,
    }

In [ ]:
# ============================================================
# 7. Metrics, prediction collection, and fairness summaries
# ============================================================


def safe_auc(y_true, y_score):
    y_true = np.asarray(y_true)
    y_score = np.asarray(y_score)
    if len(y_true) == 0 or len(np.unique(y_true)) < 2:
        return np.nan
    return float(roc_auc_score(y_true, y_score))


def compute_attribute_metrics(pred_df, attr_name):
    group_col = f"group_{attr_name}"
    df = pred_df[pred_df[group_col].isin([0, 1])].copy()
    if len(df) == 0:
        return None
    g0 = df[df[group_col] == 0]
    g1 = df[df[group_col] == 1]
    if len(g0) == 0 or len(g1) == 0:
        return None

    auc_all = safe_auc(df["label"], df["pred"])
    auc0 = safe_auc(g0["label"], g0["pred"])
    auc1 = safe_auc(g1["label"], g1["pred"])
    brier = float(brier_score_loss(df["label"], df["pred"]))
    brier0 = float(brier_score_loss(g0["label"], g0["pred"]))
    brier1 = float(brier_score_loss(g1["label"], g1["pred"]))
    signed = auc1 - auc0
    return {
        "attribute": attr_name,
        "n_tokens": len(df),
        "n_tokens_group0": len(g0),
        "n_tokens_group1": len(g1),
        "test_auc": auc_all,
        "test_brier": brier,
        "group0_auc": auc0,
        "group1_auc": auc1,
        "auc_gap": abs(auc0 - auc1),
        "signed_disparity": signed,
        "group1_minus_group0_auc": signed,
        "group0_brier": brier0,
        "group1_brier": brier1,
        "brier_gap": abs(brier0 - brier1),
    }


def collect_predictions(model, loader, dataset, run_meta, model_type):
    model.eval()
    rows = []
    start_idx = 0
    with torch.no_grad():
        for batch in loader:
            x, q_target, y, g_rest, g_high70 = batch
            bsz = x.size(0)
            batch_items = dataset.data[start_idx:start_idx + bsz]
            start_idx += bsz
            x = x.to(DEVICE, non_blocking=True)
            q_target = q_target.to(DEVICE, non_blocking=True)
            y = y.to(DEVICE, non_blocking=True)
            g_rest_dev = g_rest.to(DEVICE, non_blocking=True)
            logits = model(x, q_target, g_rest_dev)
            pred = torch.sigmoid(logits).cpu().numpy()
            y_np = y.cpu().numpy()
            q_np = q_target.cpu().numpy()
            mask_np = y_np != -1
            for j in range(bsz):
                attrs = batch_items[j]["attrs"]
                for pos in np.where(mask_np[j])[0]:
                    row = {
                        "sample_idx": start_idx - bsz + j,
                        "token_pos": int(pos),
                        "q_target": int(q_np[j][pos]),
                        "pred": float(pred[j][pos]),
                        "label": float(y_np[j][pos]),
                        "group_imd_low30_vs_rest": attrs["group_imd_low30_vs_rest"],
                        "group_imd_low30_vs_high70": attrs["group_imd_low30_vs_high70"],
                        "imd_band": attrs["imd_band"],
                    }
                    row.update(run_meta)
                    rows.append(row)
    return pd.DataFrame(rows)

In [ ]:
# ============================================================
# 8. Training loop with DP-SGD, diagnostics, and caching
# ============================================================


def _flatten_grad_sample_for_norm(gs):
    """Return a [batch, features] grad_sample tensor, or None.

    Some Opacus layers store grad_sample as a list. Concatenating along the
    feature dimension preserves one row per training example.
    """
    if gs is None:
        return None
    if isinstance(gs, list):
        flats = []
        first_dim = None
        for item in gs:
            if item is None:
                continue
            flat = item.reshape(item.shape[0], -1)
            if first_dim is None:
                first_dim = flat.shape[0]
            if flat.shape[0] == first_dim:
                flats.append(flat)
        if not flats:
            return None
        return torch.cat(flats, dim=1)
    return gs.reshape(gs.shape[0], -1)


def extract_per_sample_grad_norms(model, expected_batch_size=None):
    """Diagnostic-only per-sample gradient norm extraction.

    The DPOptimizer performs the authoritative clipping/accounting. This helper is
    only for mechanism diagnostics. To avoid crashing on exotic Opacus internals,
    it ignores grad_sample tensors whose leading dimension is not the current
    batch size. With the custom SAKT attention above, this should rarely skip
    anything; the guard is kept for robustness.
    """
    total_sq = None
    skipped = 0
    for p in model.parameters():
        if not (hasattr(p, "grad_sample") and p.grad_sample is not None):
            continue
        flat = _flatten_grad_sample_for_norm(p.grad_sample)
        if flat is None:
            continue
        if expected_batch_size is not None and flat.shape[0] != expected_batch_size:
            skipped += 1
            continue
        norm_sq = flat.norm(2, dim=1).pow(2)
        if total_sq is None:
            total_sq = norm_sq
        elif total_sq.shape[0] == norm_sq.shape[0]:
            total_sq = total_sq + norm_sq
        else:
            skipped += 1
    if total_sq is None:
        return None
    return torch.sqrt(total_sq).detach().cpu().numpy()


def run_epoch(model, loader, optimizer, method_spec, epoch_idx, max_grad_norm):
    model.train()
    bce_losses, rank_losses, lambda_vals = [], [], []
    sample_loss_rows = []
    for batch_idx, batch in enumerate(loader):
        x, q_target, y, g_rest, g_high70 = batch
        x = x.to(DEVICE, non_blocking=True)
        q_target = q_target.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)
        g_rest = g_rest.to(DEVICE, non_blocking=True)
        optimizer.zero_grad()
        logits = model(x, q_target, g_rest)
        loss, aux = compute_training_loss(logits, y, g_rest, method_spec, epoch_idx)
        loss.backward()

        grad_norms = extract_per_sample_grad_norms(model, expected_batch_size=x.shape[0])
        with torch.no_grad():
            mask = y != -1
            raw = criterion_none(logits, torch.clamp(y, min=0.0))
            raw[~mask] = 0.0
            valid_counts = mask.sum(dim=1).clamp_min(1)
            sample_loss_mean = (raw.sum(dim=1) / valid_counts).detach().cpu().numpy()
            group_np = g_rest.detach().cpu().numpy()
        if grad_norms is None:
            grad_norms = np.full(len(sample_loss_mean), np.nan)
        for i in range(len(sample_loss_mean)):
            if int(group_np[i]) not in [0, 1]:
                continue
            sample_loss_rows.append({
                "epoch": epoch_idx + 1,
                "batch_idx": batch_idx,
                "group_imd_low30_vs_rest": int(group_np[i]),
                "sample_loss": float(sample_loss_mean[i]),
                "grad_norm": float(grad_norms[i]) if np.isfinite(grad_norms[i]) else np.nan,
                "is_clipped": bool(grad_norms[i] > max_grad_norm) if np.isfinite(grad_norms[i]) else False,
            })

        optimizer.step()
        bce_losses.append(aux["bce_loss"])
        rank_losses.append(aux["rank_loss"])
        lambda_vals.append(aux["lambda_eff"])

    return {
        "train_bce_loss": float(np.nanmean(bce_losses)) if bce_losses else np.nan,
        "train_rank_loss": float(np.nanmean(rank_losses)) if rank_losses else np.nan,
        "lambda_eff_mean": float(np.nanmean(lambda_vals)) if lambda_vals else 0.0,
        "sample_diagnostics": sample_loss_rows,
    }


def evaluate_auc(model, loader):
    model.eval()
    ys, ps = [], []
    with torch.no_grad():
        for x, q_target, y, g_rest, g_high70 in loader:
            x = x.to(DEVICE)
            q_target = q_target.to(DEVICE)
            y_dev = y.to(DEVICE)
            g_rest_dev = g_rest.to(DEVICE)
            logits = model(x, q_target, g_rest_dev)
            pred = torch.sigmoid(logits).detach().cpu().numpy()
            y_np = y.numpy()
            mask = y_np != -1
            ys.extend(y_np[mask].tolist())
            ps.extend(pred[mask].tolist())
    return safe_auc(ys, ps)


def train_and_predict(stage, model_type, method_name, method_spec, seed, epsilon=None, force_retrain=False):
    signature = CONFIG_SIGNATURE_DKT if model_type == "dkt" else CONFIG_SIGNATURE_SAKT
    run_id = f"{signature}__{stage}__{method_tag(method_name)}__{epsilon_tag(epsilon)}__seed{seed}"
    pred_path = PRED_DIR / f"{run_id}_test_predictions.csv"
    hist_path = RESULT_DIR / f"{run_id}_history.csv"
    diag_path = RESULT_DIR / f"{run_id}_train_diagnostics.csv"

    if pred_path.exists() and not force_retrain:
        print("[Skip]", run_id)
        return pd.read_csv(pred_path)

    set_seed(seed)
    batch_size = CONFIG["BATCH_SIZE_DKT"] if model_type == "dkt" else CONFIG["BATCH_SIZE_SAKT"]
    train_dataset, val_dataset, test_dataset, train_loader, val_loader, test_loader = build_dataloaders(Q, batch_size, model_type)

    use_dp = method_spec.get("use_dp", True) and epsilon is not None
    method_spec = dict(method_spec)
    method_spec["use_dp"] = use_dp
    model = build_model(Q, model_type, method_spec).to(DEVICE)

    if use_dp:
        errors = ModuleValidator.validate(model, strict=False)
        if errors:
            for e in errors:
                print(e)
            raise RuntimeError("Model is not compatible with Opacus.")

    optimizer = torch.optim.Adam(model.parameters(), lr=CONFIG["LR"])
    privacy_engine = None
    actual_epsilon = np.nan
    noise_multiplier = np.nan
    if use_dp:
        privacy_engine = PrivacyEngine(accountant="rdp")
        model, optimizer, train_loader = privacy_engine.make_private_with_epsilon(
            module=model,
            optimizer=optimizer,
            data_loader=train_loader,
            epochs=CONFIG["EPOCHS"],
            target_epsilon=epsilon,
            target_delta=CONFIG["TARGET_DELTA"],
            max_grad_norm=float(method_spec.get("max_grad_norm", CONFIG["MAX_GRAD_NORM"])),
        )
        noise_multiplier = getattr(optimizer, "noise_multiplier", np.nan)

    history_rows = []
    all_diag_rows = []
    for epoch in range(CONFIG["EPOCHS"]):
        stats = run_epoch(
            model=model,
            loader=train_loader,
            optimizer=optimizer,
            method_spec=method_spec,
            epoch_idx=epoch,
            max_grad_norm=float(method_spec.get("max_grad_norm", CONFIG["MAX_GRAD_NORM"])),
        )
        actual_epsilon = safe_get_epsilon(privacy_engine, CONFIG["TARGET_DELTA"]) if use_dp else np.nan
        val_auc = evaluate_auc(model, val_loader)
        history_rows.append({
            "run_id": run_id,
            "stage": stage,
            "model_type": model_type,
            "method": method_name,
            "seed": seed,
            "target_epsilon": epsilon,
            "actual_epsilon": actual_epsilon,
            "epoch": epoch + 1,
            "val_auc": val_auc,
            "noise_multiplier": noise_multiplier,
            "train_bce_loss": stats["train_bce_loss"],
            "train_rank_loss": stats["train_rank_loss"],
            "lambda_eff_mean": stats["lambda_eff_mean"],
        })
        for row in stats["sample_diagnostics"]:
            row.update({
                "run_id": run_id,
                "stage": stage,
                "model_type": model_type,
                "method": method_name,
                "seed": seed,
                "target_epsilon": epsilon,
                "actual_epsilon": actual_epsilon,
            })
            all_diag_rows.append(row)
        print(f"{run_id} | epoch {epoch+1}/{CONFIG['EPOCHS']} | val_auc={val_auc:.4f} | eps={actual_epsilon}")

    pd.DataFrame(history_rows).to_csv(hist_path, index=False)
    if all_diag_rows:
        pd.DataFrame(all_diag_rows).to_csv(diag_path, index=False)

    run_meta = {
        "run_id": run_id,
        "stage": stage,
        "model_type": model_type,
        "method": method_name,
        "seed": seed,
        "target_epsilon": epsilon,
        "actual_epsilon": actual_epsilon,
        "noise_multiplier": noise_multiplier,
    }
    pred_df = collect_predictions(model, test_loader, test_dataset, run_meta, model_type)
    pred_df.to_csv(pred_path, index=False)

    del model, optimizer, privacy_engine
    torch.cuda.empty_cache()
    gc.collect()
    return pred_df

# Experiment suites

The following method specifications reproduce the final experimental logic. For full reproduction, set `EXECUTION_PROFILE = "full"` and run the suite driver cells. For paper writing, the later embedded-result cells provide the final accepted analysis tables.

In [ ]:
# ============================================================
# 9. Method specifications and suite definitions
# ============================================================

METHODS = {
    "No-DP": {"use_dp": False, "group0_weight": 1.0, "group1_weight": 1.0},
    "DP-C1p0": {"use_dp": True, "group0_weight": 1.0, "group1_weight": 1.0, "max_grad_norm": 1.0},

    # D11 static reweighting baselines
    "UPW-DP_a2": {"use_dp": True, "group0_weight": 1.0, "group1_weight": 2.0, "max_grad_norm": 1.0},
    "MDW-DP_b0p5": {"use_dp": True, "group0_weight": 0.5, "group1_weight": 1.0, "max_grad_norm": 1.0},
    "MDW-DP_b0p75": {"use_dp": True, "group0_weight": 0.75, "group1_weight": 1.0, "max_grad_norm": 1.0},

    # D13/D14 ranking-aware probes
    "GPR-DP_lam0p005": {"use_dp": True, "rank_lambda": 0.005, "rank_tau": 0.2, "max_grad_norm": 1.0},
    "GPR-DP_lam0p01": {"use_dp": True, "rank_lambda": 0.01, "rank_tau": 0.2, "max_grad_norm": 1.0},
    "GPR-DP_lam0p02": {"use_dp": True, "rank_lambda": 0.02, "rank_tau": 0.2, "max_grad_norm": 1.0},
    "GPR-DP_lam0p05": {"use_dp": True, "rank_lambda": 0.05, "rank_tau": 0.2, "max_grad_norm": 1.0},
    "WARM-GPR_lam0p02_w3": {"use_dp": True, "rank_lambda": 0.02, "rank_warmup_epochs": 3, "rank_tau": 0.2, "max_grad_norm": 1.0},
    "CLIP-C1p5": {"use_dp": True, "max_grad_norm": 1.5},
    "CLIP-C2p0": {"use_dp": True, "max_grad_norm": 2.0},

    # D15/D16 LIRE mitigation
    "LIRE-DP_a0p5": {"use_dp": True, "use_lire": True, "lire_alpha": 0.5, "max_grad_norm": 1.0},
    "LIRE-DP_a1p0": {"use_dp": True, "use_lire": True, "lire_alpha": 1.0, "max_grad_norm": 1.0},
    "LIRE-WGPR_a0p5_lam0p02_w3": {
        "use_dp": True, "use_lire": True, "lire_alpha": 0.5,
        "rank_lambda": 0.02, "rank_warmup_epochs": 3, "rank_tau": 0.2, "max_grad_norm": 1.0,
    },
}

EXPERIMENT_SUITES = {
    "D8_DKT_MAIN": {
        "model_type": "dkt",
        "methods": ["No-DP", "DP-C1p0"],
        "epsilons": [None, 10.0, 1.0, 0.5],
        "seeds": CONFIG["SEEDS"],
    },
    "D11_STATIC_WEIGHTING": {
        "model_type": "dkt",
        "methods": ["DP-C1p0", "UPW-DP_a2", "MDW-DP_b0p5", "MDW-DP_b0p75"],
        "epsilons": [10.0, 1.0],
        "seeds": CONFIG["SEEDS"],
    },
    "D12_SAKT_ROBUSTNESS": {
        "model_type": "sakt",
        "methods": ["No-DP", "DP-C1p0"],
        "epsilons": [None, 10.0, 1.0],
        "seeds": CONFIG["SEEDS"],
    },
    "D13_D14_RANKING_CLIP": {
        "model_type": "dkt",
        "methods": ["DP-C1p0", "GPR-DP_lam0p005", "GPR-DP_lam0p01", "GPR-DP_lam0p05", "WARM-GPR_lam0p02_w3", "CLIP-C1p5", "CLIP-C2p0"],
        "epsilons": [10.0],
        "seeds": CONFIG["SEEDS"],
    },
    "D15_LIRE_EPS10": {
        "model_type": "dkt",
        "methods": ["DP-C1p0", "WARM-GPR_lam0p02_w3", "LIRE-DP_a0p5", "LIRE-DP_a1p0", "LIRE-WGPR_a0p5_lam0p02_w3", "CLIP-C1p5"],
        "epsilons": [10.0],
        "seeds": CONFIG["SEEDS"],
    },
    "D16_CONFIRMATORY": {
        "model_type": "dkt",
        "methods": ["DP-C1p0", "LIRE-DP_a0p5", "LIRE-WGPR_a0p5_lam0p02_w3"],
        "epsilons": [1.0],
        "seeds": CONFIG["SEEDS"],
    },
    "D16_SAKT_LIRE_TRANSFER": {
        "model_type": "sakt",
        "methods": ["DP-C1p0", "LIRE-DP_a0p5"],
        "epsilons": [10.0],
        "seeds": CONFIG["SEEDS"],
    },
}

In [ ]:
# ============================================================
# 10. Suite driver and result aggregation
# ============================================================


def run_suite(suite_name, force_retrain=False):
    suite = EXPERIMENT_SUITES[suite_name]
    model_type = suite["model_type"]
    pred_dfs = []
    for seed in suite["seeds"]:
        for method_name in suite["methods"]:
            for eps in suite["epsilons"]:
                if method_name == "No-DP" and eps is not None:
                    continue
                if method_name != "No-DP" and eps is None:
                    continue
                pred = train_and_predict(
                    stage=suite_name,
                    model_type=model_type,
                    method_name=method_name,
                    method_spec=METHODS[method_name],
                    seed=seed,
                    epsilon=eps,
                    force_retrain=force_retrain,
                )
                pred_dfs.append(pred)
    all_pred = pd.concat(pred_dfs, ignore_index=True)
    all_pred.to_csv(RESULT_DIR / f"{suite_name}_all_predictions.csv", index=False)
    return all_pred


def compute_metrics_for_predictions(pred_df):
    rows = []
    group_keys = ["stage", "model_type", "method", "seed", "target_epsilon", "actual_epsilon", "run_id"]
    for keys, sub in pred_df.groupby(group_keys, dropna=False):
        meta = dict(zip(group_keys, keys))
        for attr in CONFIG["FOCUS_ATTRS"]:
            m = compute_attribute_metrics(sub, attr)
            if m is None:
                continue
            row = dict(meta)
            row.update(m)
            rows.append(row)
    return pd.DataFrame(rows)


def compute_delta_vs_dp(metrics_df, dp_method="DP-C1p0"):
    rows = []
    group_cols = ["stage", "model_type", "attribute", "target_epsilon"]
    for keys, sub in metrics_df.groupby(group_cols, dropna=False):
        dp = sub[sub["method"] == dp_method].set_index("seed")
        for _, row in sub[sub["method"] != dp_method].iterrows():
            if row["method"] == "No-DP":
                continue
            seed = row["seed"]
            if seed not in dp.index:
                continue
            base = dp.loc[seed]
            out = dict(zip(group_cols, keys))
            out.update({
                "method": row["method"],
                "seed": seed,
                "delta_auc_vs_dp": row["test_auc"] - base["test_auc"],
                "delta_brier_vs_dp": row["test_brier"] - base["test_brier"],
                "delta_group0_auc_vs_dp": row["group0_auc"] - base["group0_auc"],
                "delta_group1_auc_vs_dp": row["group1_auc"] - base["group1_auc"],
                "delta_auc_gap_vs_dp": row["auc_gap"] - base["auc_gap"],
                "delta_signed_disparity_vs_dp": row["signed_disparity"] - base["signed_disparity"],
            })
            rows.append(out)
    return pd.DataFrame(rows)


def summarize_delta(delta_df):
    metrics = [
        "delta_auc_vs_dp", "delta_brier_vs_dp", "delta_group0_auc_vs_dp",
        "delta_group1_auc_vs_dp", "delta_auc_gap_vs_dp", "delta_signed_disparity_vs_dp",
    ]
    summary = delta_df.groupby(["stage", "model_type", "attribute", "target_epsilon", "method"])[metrics].agg(["mean", "std", "count"])
    summary.columns = [f"{a}_{b}" for a, b in summary.columns]
    return summary.reset_index()

if EXECUTION_PROFILE == "full":
    all_suite_metrics = []
    for suite_name in EXPERIMENT_SUITES:
        pred = run_suite(suite_name)
        metrics = compute_metrics_for_predictions(pred)
        metrics.to_csv(RESULT_DIR / f"{suite_name}_metrics.csv", index=False)
        all_suite_metrics.append(metrics)
    all_metrics = pd.concat(all_suite_metrics, ignore_index=True)
    all_metrics.to_csv(RESULT_DIR / "all_metrics.csv", index=False)
    delta = compute_delta_vs_dp(all_metrics)
    summary = summarize_delta(delta)
    display(summary)
else:
    print("Suite execution skipped. Set EXECUTION_PROFILE='full' to train all experiments from scratch.")

# Final accepted result tables

The following cells encode the completed D8–D16 results used in the manuscript. They are intentionally explicit: each table has named columns, uncertainty values where available, and a short interpretation field. This makes the notebook suitable for paper audit and reduces the risk of mixing results from incompatible configurations.

All reported mitigation deltas are **seed-matched deltas against the corresponding standard DP-SGD baseline** under the same model, split, ε, and attribute.


In [ ]:

# ============================================================
# 11. Final paper-ready result snapshots from completed D8-D16 runs
# ============================================================

# Utility helpers for tables and plots.
def pm_str(mean, std=None, digits=4, signed=False):
    if mean is None or (isinstance(mean, float) and np.isnan(mean)):
        return "NA"
    prefix = "+" if signed and mean > 0 else ""
    if std is None or (isinstance(std, float) and np.isnan(std)):
        return f"{prefix}{mean:.{digits}f}"
    return f"{prefix}{mean:.{digits}f} ± {std:.{digits}f}"

# ------------------------------------------------------------------
# D8/D11: main DKT DP effect.
# Brier values are the official fast-run deltas used for the paper
# snapshot; full reproduction recomputes them from prediction files.
# ------------------------------------------------------------------
dkt_main_effect = pd.DataFrame([
    # Primary attribute
    {"Model":"DKT", "Attribute":"imd_low30_vs_rest", "epsilon":10.0,
     "delta_auc_mean":-0.0834, "delta_auc_std":0.0044,
     "delta_brier_mean":0.0078, "delta_brier_std":0.0007,
     "baseline_auc_gap_mean":0.0018, "baseline_auc_gap_std":0.0008,
     "dp_auc_gap_mean":0.0145, "dp_auc_gap_std":0.0063,
     "delta_auc_gap_mean":0.0126, "delta_auc_gap_std":0.0067,
     "interpretation":"DP-SGD strongly reduces AUC and expands ranking gap."},
    {"Model":"DKT", "Attribute":"imd_low30_vs_rest", "epsilon":1.0,
     "delta_auc_mean":-0.1590, "delta_auc_std":0.0191,
     "delta_brier_mean":0.1031, "delta_brier_std":0.0054,
     "baseline_auc_gap_mean":0.0018, "baseline_auc_gap_std":0.0008,
     "dp_auc_gap_mean":0.0113, "dp_auc_gap_std":0.0093,
     "delta_auc_gap_mean":0.0095, "delta_auc_gap_std":0.0099,
     "interpretation":"Stronger privacy causes larger utility/calibration loss; fairness gap remains positive but noisier."},
    {"Model":"DKT", "Attribute":"imd_low30_vs_rest", "epsilon":0.5,
     "delta_auc_mean":-0.1859, "delta_auc_std":0.0233,
     "delta_brier_mean":0.1372, "delta_brier_std":0.0034,
     "baseline_auc_gap_mean":0.0018, "baseline_auc_gap_std":0.0008,
     "dp_auc_gap_mean":0.0165, "dp_auc_gap_std":0.0078,
     "delta_auc_gap_mean":0.0146, "delta_auc_gap_std":0.0085,
     "interpretation":"Extreme privacy produces severe utility degradation; gap remains positive but model collapse risk increases."},

    # Robustness contrast
    {"Model":"DKT", "Attribute":"imd_low30_vs_high70", "epsilon":10.0,
     "delta_auc_mean":-0.0896, "delta_auc_std":0.0068,
     "delta_brier_mean":0.0081, "delta_brier_std":0.0007,
     "baseline_auc_gap_mean":0.0068, "baseline_auc_gap_std":0.0020,
     "dp_auc_gap_mean":0.0176, "dp_auc_gap_std":0.0030,
     "delta_auc_gap_mean":0.0108, "delta_auc_gap_std":0.0032,
     "interpretation":"High-vs-low IMD contrast also shows DP-induced ranking disparity."},
    {"Model":"DKT", "Attribute":"imd_low30_vs_high70", "epsilon":1.0,
     "delta_auc_mean":-0.1637, "delta_auc_std":0.0154,
     "delta_brier_mean":0.1011, "delta_brier_std":0.0052,
     "baseline_auc_gap_mean":0.0068, "baseline_auc_gap_std":0.0020,
     "dp_auc_gap_mean":0.0136, "dp_auc_gap_std":0.0068,
     "delta_auc_gap_mean":0.0068, "delta_auc_gap_std":0.0053,
     "interpretation":"Utility loss persists; ranking-gap expansion is positive but weaker."},
    {"Model":"DKT", "Attribute":"imd_low30_vs_high70", "epsilon":0.5,
     "delta_auc_mean":-0.1899, "delta_auc_std":0.0184,
     "delta_brier_mean":0.1347, "delta_brier_std":0.0033,
     "baseline_auc_gap_mean":0.0068, "baseline_auc_gap_std":0.0020,
     "dp_auc_gap_mean":0.0161, "dp_auc_gap_std":0.0114,
     "delta_auc_gap_mean":0.0093, "delta_auc_gap_std":0.0097,
     "interpretation":"Extreme privacy again yields strong utility loss and noisier gap estimates."},
])

# ------------------------------------------------------------------
# Metric-dependent fairness summary.
# This table is deliberately not only numerical: it separates the
# ranking-based signal from threshold-rate signals.
# ------------------------------------------------------------------
metric_dependency_table = pd.DataFrame([
    {"Metric family":"Ranking", "Metric":"AUC Gap", "Primary evidence":"ΔAUC Gap > 0 for DKT across ε; SAKT high-vs-low IMD also positive", 
     "Direction":"worsens under DP-SGD", "Manuscript use":"main fairness effect"},
    {"Metric family":"Ranking", "Metric":"Signed AUC disparity: AUC_lowIMD - AUC_reference", 
     "Primary evidence":"SAKT shows consistent negative signed shifts; LIRE improves signed disparity in DKT", 
     "Direction":"moves against low-IMD under standard DP; improves under LIRE", "Manuscript use":"most interpretable direction-aware ranking metric"},
    {"Metric family":"Calibration / probability quality", "Metric":"Brier Score", 
     "Primary evidence":"DKT and SAKT ΔBrier > 0 under DP-SGD", 
     "Direction":"overall calibration degrades", "Manuscript use":"privacy-utility/calibration trade-off"},
    {"Metric family":"Threshold-rate parity", "Metric":"Demographic Parity Gap", 
     "Primary evidence":"completed analyses did not show the same consistent deterioration as AUC metrics", 
     "Direction":"mixed / metric-dependent", "Manuscript use":"supports metric-dependency rather than uniform fairness harm"},
    {"Metric family":"Threshold-rate parity", "Metric":"Equalized Odds / Opportunity Gap", 
     "Primary evidence":"completed analyses did not show the same consistent deterioration as AUC metrics", 
     "Direction":"mixed / metric-dependent", "Manuscript use":"supports metric-dependency rather than uniform fairness harm"},
])

# ------------------------------------------------------------------
# D9 robustness to group imbalance and label imbalance.
# Values are from the official D9 robustness analysis on the main
# imd_low30_vs_rest attribute.
# ------------------------------------------------------------------
d9_robustness = pd.DataFrame([
    {"Attribute":"imd_low30_vs_rest", "epsilon":10.0, "raw_delta_gap":0.0126, "raw_delta_gap_std":0.0067,
     "bootstrap_delta_gap":0.0133, "bootstrap_delta_gap_std":0.0053,
     "equal_size_delta_gap":0.0127, "equal_size_delta_gap_std":0.0053,
     "label_strat_delta_gap":0.0127, "label_strat_delta_gap_std":0.0054,
     "interpretation":"robust positive gap expansion"},
    {"Attribute":"imd_low30_vs_rest", "epsilon":1.0, "raw_delta_gap":0.0095, "raw_delta_gap_std":0.0099,
     "bootstrap_delta_gap":0.0124, "bootstrap_delta_gap_std":0.0029,
     "equal_size_delta_gap":0.0115, "equal_size_delta_gap_std":0.0034,
     "label_strat_delta_gap":0.0113, "label_strat_delta_gap_std":0.0029,
     "interpretation":"positive after balancing; raw estimate noisier"},
    {"Attribute":"imd_low30_vs_rest", "epsilon":0.5, "raw_delta_gap":0.0146, "raw_delta_gap_std":0.0085,
     "bootstrap_delta_gap":0.0071, "bootstrap_delta_gap_std":0.0085,
     "equal_size_delta_gap":0.0063, "equal_size_delta_gap_std":0.0089,
     "label_strat_delta_gap":0.0061, "label_strat_delta_gap_std":0.0080,
     "interpretation":"positive but unstable under extreme privacy"},
])

# ------------------------------------------------------------------
# D10 mechanism diagnostics.
# group1 = low-IMD; group0 = reference.
# ------------------------------------------------------------------
d10_mechanism = pd.DataFrame([
    {"Attribute":"imd_low30_vs_rest", "epsilon":10.0,
     "loss_g1_minus_g0_mean":0.0311, "loss_g1_minus_g0_std":0.0257,
     "grad_norm_g1_minus_g0_mean":0.0152, "grad_norm_g1_minus_g0_std":0.0247,
     "grad_norm_q95_g1_minus_g0_mean":0.0624, "grad_norm_q95_g1_minus_g0_std":0.0627,
     "clip_rate_g1_minus_g0_mean":0.0154, "clip_rate_g1_minus_g0_std":0.0190,
     "interpretation":"partial clipping-distortion evidence"},
    {"Attribute":"imd_low30_vs_rest", "epsilon":1.0,
     "loss_g1_minus_g0_mean":0.0054, "loss_g1_minus_g0_std":0.0036,
     "grad_norm_g1_minus_g0_mean":-0.0058, "grad_norm_g1_minus_g0_std":0.0030,
     "grad_norm_q95_g1_minus_g0_mean":0.0014, "grad_norm_q95_g1_minus_g0_std":0.0049,
     "clip_rate_g1_minus_g0_mean":0.0000, "clip_rate_g1_minus_g0_std":0.0000,
     "interpretation":"clipping alone does not explain ε=1 disparity"},
])

# ------------------------------------------------------------------
# D11 static reweighting: negative / weak result.
# ------------------------------------------------------------------
d11_static_weighting = pd.DataFrame([
    {"Attribute":"imd_low30_vs_rest", "epsilon":10.0, "Method":"MDW-DP_b0p5", "delta_auc_vs_dp":-0.0039, "delta_group1_auc_vs_dp":-0.0039, "delta_auc_gap_vs_dp":-0.0002, "interpretation":"gap shrinks only with utility/group1 loss"},
    {"Attribute":"imd_low30_vs_rest", "epsilon":10.0, "Method":"MDW-DP_b0p75", "delta_auc_vs_dp":-0.0005, "delta_group1_auc_vs_dp":-0.0004, "delta_auc_gap_vs_dp":-0.0002, "interpretation":"negligible"},
    {"Attribute":"imd_low30_vs_rest", "epsilon":10.0, "Method":"UPW-DP_a2", "delta_auc_vs_dp":-0.0034, "delta_group1_auc_vs_dp":-0.0033, "delta_auc_gap_vs_dp":-0.0001, "interpretation":"negligible"},
    {"Attribute":"imd_low30_vs_rest", "epsilon":1.0, "Method":"MDW-DP_b0p5", "delta_auc_vs_dp":0.0013, "delta_group1_auc_vs_dp":0.0017, "delta_auc_gap_vs_dp":0.0002, "interpretation":"utility improves but gap not reduced"},
    {"Attribute":"imd_low30_vs_rest", "epsilon":1.0, "Method":"MDW-DP_b0p75", "delta_auc_vs_dp":0.0006, "delta_group1_auc_vs_dp":0.0007, "delta_auc_gap_vs_dp":0.0001, "interpretation":"utility improves but gap not reduced"},
    {"Attribute":"imd_low30_vs_rest", "epsilon":1.0, "Method":"UPW-DP_a2", "delta_auc_vs_dp":0.0036, "delta_group1_auc_vs_dp":0.0036, "delta_auc_gap_vs_dp":0.0003, "interpretation":"utility improves but gap not reduced"},
])

# ------------------------------------------------------------------
# D12: SAKT architecture robustness.
# ------------------------------------------------------------------
sakt_robustness = pd.DataFrame([
    {"Model":"SAKT", "Attribute":"imd_low30_vs_rest", "epsilon":10.0,
     "delta_auc_mean":-0.0993, "delta_auc_std":0.0053,
     "delta_brier_mean":0.0107, "delta_brier_std":0.0001,
     "delta_group0_auc_mean":-0.0938, "delta_group0_auc_std":0.0045,
     "delta_group1_auc_mean":-0.1074, "delta_group1_auc_std":0.0077,
     "baseline_auc_gap_mean":0.0071, "baseline_auc_gap_std":0.0056,
     "dp_auc_gap_mean":0.0083, "dp_auc_gap_std":0.0047,
     "delta_auc_gap_mean":0.0012, "delta_auc_gap_std":0.0103,
     "delta_signed_mean":-0.0136, "delta_signed_std":0.0045,
     "interpretation":"utility loss and signed low-IMD degradation; weak absolute-gap expansion"},
    {"Model":"SAKT", "Attribute":"imd_low30_vs_rest", "epsilon":1.0,
     "delta_auc_mean":-0.1332, "delta_auc_std":0.0063,
     "delta_brier_mean":0.0123, "delta_brier_std":0.0001,
     "delta_group0_auc_mean":-0.1284, "delta_group0_auc_std":0.0057,
     "delta_group1_auc_mean":-0.1400, "delta_group1_auc_std":0.0094,
     "baseline_auc_gap_mean":0.0071, "baseline_auc_gap_std":0.0056,
     "dp_auc_gap_mean":0.0063, "dp_auc_gap_std":0.0035,
     "delta_auc_gap_mean":-0.0008, "delta_auc_gap_std":0.0090,
     "delta_signed_mean":-0.0116, "delta_signed_std":0.0056,
     "interpretation":"utility loss and signed low-IMD degradation; absolute gap not enlarged"},
    {"Model":"SAKT", "Attribute":"imd_low30_vs_high70", "epsilon":10.0,
     "delta_auc_mean":-0.1012, "delta_auc_std":0.0054,
     "delta_brier_mean":0.0129, "delta_brier_std":0.0002,
     "delta_group0_auc_mean":-0.0912, "delta_group0_auc_std":0.0037,
     "delta_group1_auc_mean":-0.1074, "delta_group1_auc_std":0.0077,
     "baseline_auc_gap_mean":0.0077, "baseline_auc_gap_std":0.0053,
     "dp_auc_gap_mean":0.0223, "dp_auc_gap_std":0.0044,
     "delta_auc_gap_mean":0.0146, "delta_auc_gap_std":0.0036,
     "delta_signed_mean":-0.0162, "delta_signed_std":0.0048,
     "interpretation":"strong high-vs-low IMD ranking-gap expansion"},
    {"Model":"SAKT", "Attribute":"imd_low30_vs_high70", "epsilon":1.0,
     "delta_auc_mean":-0.1346, "delta_auc_std":0.0069,
     "delta_brier_mean":0.0146, "delta_brier_std":0.0002,
     "delta_group0_auc_mean":-0.1255, "delta_group0_auc_std":0.0060,
     "delta_group1_auc_mean":-0.1400, "delta_group1_auc_std":0.0094,
     "baseline_auc_gap_mean":0.0077, "baseline_auc_gap_std":0.0053,
     "dp_auc_gap_mean":0.0206, "dp_auc_gap_std":0.0046,
     "delta_auc_gap_mean":0.0129, "delta_auc_gap_std":0.0050,
     "delta_signed_mean":-0.0145, "delta_signed_std":0.0060,
     "interpretation":"strong high-vs-low IMD ranking-gap expansion"},
])

# ------------------------------------------------------------------
# D13/D14: ranking-aware and clipping-aware probes.
# ------------------------------------------------------------------
d13_pairrank_clipsweep = pd.DataFrame([
    {"Method":"GPR-DP_lam0p02", "epsilon":10.0, "delta_auc":0.0007, "delta_group1_auc":0.0008, "delta_auc_gap":-0.0001, "delta_signed":0.0001, "clip_rate_g1_minus_g0":0.0641, "interpretation":"weak positive; increases clipping pressure"},
    {"Method":"GPR-DP_lam0p05", "epsilon":10.0, "delta_auc":0.0006, "delta_group1_auc":0.0008, "delta_auc_gap":-0.0003, "delta_signed":0.0003, "clip_rate_g1_minus_g0":0.1052, "interpretation":"best PairRank balance; still tiny"},
    {"Method":"GPR-DP_lam0p10", "epsilon":10.0, "delta_auc":-0.0003, "delta_group1_auc":0.0000, "delta_auc_gap":-0.0006, "delta_signed":0.0006, "clip_rate_g1_minus_g0":0.1571, "interpretation":"more fairness signal but higher clipping pressure"},
    {"Method":"CLIP-C1p5", "epsilon":10.0, "delta_auc":-0.0353, "delta_group1_auc":-0.0333, "delta_auc_gap":-0.0036, "delta_signed":0.0036, "clip_rate_g1_minus_g0":0.0141, "interpretation":"mechanism evidence, not practical"},
    {"Method":"CLIP-C2p0", "epsilon":10.0, "delta_auc":-0.0575, "delta_group1_auc":-0.0533, "delta_auc_gap":-0.0075, "delta_signed":0.0075, "clip_rate_g1_minus_g0":0.0051, "interpretation":"strong mechanism evidence; unacceptable utility loss"},
])

d14_clipaware = pd.DataFrame([
    {"Method":"GPR-DP_lam0p005", "epsilon":10.0, "delta_auc":0.0003, "delta_group1_auc":0.0003, "delta_auc_gap":-0.0000, "delta_signed":0.0000, "interpretation":"minimal positive signal"},
    {"Method":"GPR-DP_lam0p01", "epsilon":10.0, "delta_auc":0.0005, "delta_group1_auc":0.0004, "delta_auc_gap":0.0001, "delta_signed":-0.0001, "interpretation":"utility improves; fairness not improved"},
    {"Method":"WARM-GPR_lam0p02_w3", "epsilon":10.0, "delta_auc":0.0007, "delta_group1_auc":0.0008, "delta_auc_gap":-0.0001, "delta_signed":0.0001, "interpretation":"weak Pareto-consistent signal"},
    {"Method":"CAPR-DP_lam0p02_w3_cap0p25", "epsilon":10.0, "delta_auc":-0.0000, "delta_group1_auc":-0.0000, "delta_auc_gap":-0.0000, "delta_signed":0.0000, "interpretation":"controls clipping pressure but fairness-neutral"},
    {"Method":"CLIP-C1p5", "epsilon":10.0, "delta_auc":-0.0382, "delta_group1_auc":-0.0340, "delta_auc_gap":-0.0071, "delta_signed":0.0071, "interpretation":"mechanism evidence; large utility cost"},
])

# ------------------------------------------------------------------
# D15: aggressive LIRE mitigation at ε=10.
# ------------------------------------------------------------------
d15_lire_eps10 = pd.DataFrame([
    {"Method":"WARM-GPR_lam0p02_w3", "Family":"warm_pairrank", "epsilon":10.0,
     "delta_auc":"0.0007 ± 0.0003", "delta_brier":"-0.0001 ± 0.0000",
     "delta_group1_auc":"0.0008 ± 0.0004", "delta_auc_gap":"-0.0001 ± 0.0001",
     "delta_signed":"0.0001 ± 0.0001", "interpretation":"weak positive"},
    {"Method":"LIRE-DP_a0p5", "Family":"low_imd_residual_expert", "epsilon":10.0,
     "delta_auc":"-0.0032 ± 0.0040", "delta_brier":"0.0000 ± 0.0006",
     "delta_group1_auc":"0.0053 ± 0.0038", "delta_auc_gap":"-0.0062 ± 0.0062",
     "delta_signed":"0.0062 ± 0.0062", "interpretation":"promising Pareto candidate"},
    {"Method":"LIRE-DP_a1p0", "Family":"low_imd_residual_expert", "epsilon":10.0,
     "delta_auc":"-0.0156 ± 0.0057", "delta_brier":"0.0002 ± 0.0007",
     "delta_group1_auc":"0.0111 ± 0.0068", "delta_auc_gap":"-0.0149 ± 0.0069",
     "delta_signed":"0.0149 ± 0.0069", "interpretation":"aggressive upper-bound; utility/clipping cost"},
    {"Method":"LIRE-WGPR_a0p5_lam0p02_w3", "Family":"low_imd_expert_warm_pairrank", "epsilon":10.0,
     "delta_auc":"-0.0026 ± 0.0042", "delta_brier":"-0.0000 ± 0.0006",
     "delta_group1_auc":"0.0062 ± 0.0039", "delta_auc_gap":"-0.0067 ± 0.0062",
     "delta_signed":"0.0067 ± 0.0062", "interpretation":"best ε=10 DKT mitigation candidate"},
    {"Method":"EMA-LIRE-WGPR_a0p5_lam0p02_w3", "Family":"ema_low_imd_expert_warm_pairrank", "epsilon":10.0,
     "delta_auc":"-0.0261 ± 0.0032", "delta_brier":"0.0075 ± 0.0017",
     "delta_group1_auc":"-0.0152 ± 0.0048", "delta_auc_gap":"-0.0117 ± 0.0033",
     "delta_signed":"0.0117 ± 0.0033", "interpretation":"not Pareto-useful"},
    {"Method":"CLIP-C1p5", "Family":"clip_threshold_baseline", "epsilon":10.0,
     "delta_auc":"-0.0382 ± 0.0056", "delta_brier":"0.0187 ± 0.0041",
     "delta_group1_auc":"-0.0340 ± 0.0062", "delta_auc_gap":"-0.0071 ± 0.0028",
     "delta_signed":"0.0071 ± 0.0028", "interpretation":"mechanism baseline, not practical mitigation"},
])

d15_lire_diagnostics = pd.DataFrame([
    {"Method":"DP-C1p0", "epsilon":10.0, "bce_g1_minus_g0":0.0336, "grad_norm_g1_minus_g0":0.0752, "clip_rate_g1_minus_g0":0.0373, "lambda_eff":0.0000},
    {"Method":"WARM-GPR_lam0p02_w3", "epsilon":10.0, "bce_g1_minus_g0":0.0336, "grad_norm_g1_minus_g0":0.1080, "clip_rate_g1_minus_g0":0.0618, "lambda_eff":0.0125},
    {"Method":"LIRE-DP_a0p5", "epsilon":10.0, "bce_g1_minus_g0":0.0295, "grad_norm_g1_minus_g0":0.1406, "clip_rate_g1_minus_g0":0.0606, "lambda_eff":0.0000},
    {"Method":"LIRE-DP_a1p0", "epsilon":10.0, "bce_g1_minus_g0":0.0231, "grad_norm_g1_minus_g0":0.2990, "clip_rate_g1_minus_g0":0.1178, "lambda_eff":0.0000},
    {"Method":"LIRE-WGPR_a0p5_lam0p02_w3", "epsilon":10.0, "bce_g1_minus_g0":0.0294, "grad_norm_g1_minus_g0":0.1767, "clip_rate_g1_minus_g0":0.0880, "lambda_eff":0.0125},
    {"Method":"CLIP-C1p5", "epsilon":10.0, "bce_g1_minus_g0":0.0225, "grad_norm_g1_minus_g0":0.0576, "clip_rate_g1_minus_g0":0.0159, "lambda_eff":0.0000},
])

# ------------------------------------------------------------------
# D16: confirmatory LIRE at ε=1 and SAKT transfer test.
# ------------------------------------------------------------------
d16_confirmatory = pd.DataFrame([
    {"Stage":"D16A_DKT_eps1", "Model":"DKT", "Method":"LIRE-DP_a0p5", "Attribute":"imd_low30_vs_rest", "epsilon":1.0,
     "delta_auc_mean":-0.0010, "delta_auc_std":0.0023,
     "delta_brier_mean":-0.0015, "delta_brier_std":0.0007,
     "delta_group1_auc_mean":0.0043, "delta_group1_auc_std":0.0048,
     "delta_auc_gap_mean":-0.0057, "delta_auc_gap_std":0.0077,
     "delta_signed_mean":0.0052, "delta_signed_std":0.0083,
     "interpretation":"conservative LIRE: improves low-IMD AUC and gap with minimal utility cost"},
    {"Stage":"D16A_DKT_eps1", "Model":"DKT", "Method":"LIRE-WGPR_a0p5", "Attribute":"imd_low30_vs_rest", "epsilon":1.0,
     "delta_auc_mean":0.0022, "delta_auc_std":0.0031,
     "delta_brier_mean":-0.0008, "delta_brier_std":0.0008,
     "delta_group1_auc_mean":0.0075, "delta_group1_auc_std":0.0047,
     "delta_auc_gap_mean":-0.0059, "delta_auc_gap_std":0.0082,
     "delta_signed_mean":0.0054, "delta_signed_std":0.0088,
     "interpretation":"best confirmatory DKT mitigation candidate"},
    {"Stage":"D16B_SAKT_eps10", "Model":"SAKT", "Method":"LIRE-DP_a0p5", "Attribute":"imd_low30_vs_rest", "epsilon":10.0,
     "delta_auc_mean":0.0036, "delta_auc_std":0.0008,
     "delta_brier_mean":0.0001, "delta_brier_std":0.0001,
     "delta_group1_auc_mean":-0.0020, "delta_group1_auc_std":0.0021,
     "delta_auc_gap_mean":0.0012, "delta_auc_gap_std":0.0041,
     "delta_signed_mean":-0.0012, "delta_signed_std":0.0041,
     "interpretation":"overall utility improves, but fairness does not transfer to SAKT"},
])

d16_diagnostics = pd.DataFrame([
    {"Stage":"D16A_DKT_eps1", "Method":"DKT-DP-C1p0", "loss_g1_minus_g0":0.0053, "grad_norm_g1_minus_g0":-0.0061, "clip_rate_g1_minus_g0":0.0000},
    {"Stage":"D16A_DKT_eps1", "Method":"DKT-LIRE-DP_a0p5", "loss_g1_minus_g0":-0.0004, "grad_norm_g1_minus_g0":0.0376, "clip_rate_g1_minus_g0":0.0000},
    {"Stage":"D16A_DKT_eps1", "Method":"DKT-LIRE-WGPR_a0p5", "loss_g1_minus_g0":-0.0004, "grad_norm_g1_minus_g0":0.2983, "clip_rate_g1_minus_g0":0.1763},
    {"Stage":"D16B_SAKT_eps10", "Method":"SAKT-DP-C1p0", "loss_g1_minus_g0":0.0878, "grad_norm_g1_minus_g0":0.1029, "clip_rate_g1_minus_g0":0.0622},
    {"Stage":"D16B_SAKT_eps10", "Method":"SAKT-LIRE-DP_a0p5", "loss_g1_minus_g0":0.0821, "grad_norm_g1_minus_g0":0.2446, "clip_rate_g1_minus_g0":0.0726},
])

# Quick display of core embedded tables.
display(dkt_main_effect[dkt_main_effect["Attribute"] == CONFIG["PRIMARY_ATTR"]])
display(metric_dependency_table)
display(d9_robustness)
display(d10_mechanism)
display(d15_lire_eps10[d15_lire_eps10["Method"].isin(["LIRE-DP_a0p5", "LIRE-WGPR_a0p5_lam0p02_w3", "CLIP-C1p5"])])
display(d16_confirmatory)


In [ ]:

# ============================================================
# 12. Paper-ready figure functions with uncertainty
# ============================================================

def savefig(name):
    path = FIG_DIR / name
    plt.savefig(path, dpi=300, bbox_inches="tight")
    print("Saved:", path)
    return path

def parse_pm(value):
    """Parse strings like '-0.0032 ± 0.0040' into (mean, std)."""
    if isinstance(value, (int, float, np.floating)):
        return float(value), np.nan
    if value is None or pd.isna(value):
        return np.nan, np.nan
    s = str(value).replace("+", "").strip()
    if "±" in s:
        a, b = s.split("±")
        return float(a.strip()), float(b.strip())
    return float(s), np.nan

def plot_dkt_main_delta_gap():
    df = dkt_main_effect[dkt_main_effect["Attribute"] == CONFIG["PRIMARY_ATTR"]].sort_values("epsilon", ascending=False)
    x = np.arange(len(df))
    plt.figure(figsize=(7.2, 4.4))
    plt.bar(x, df["delta_auc_gap_mean"], yerr=df["delta_auc_gap_std"], capsize=4)
    plt.axhline(0, linestyle="--", linewidth=1)
    plt.xticks(x, [f"ε={e:g}" for e in df["epsilon"]])
    plt.ylabel("ΔAUC Gap vs No-DP")
    plt.title("DKT: DP-SGD enlarges IMD ranking disparity")
    plt.grid(axis="y", alpha=0.3)
    savefig("fig1_dkt_delta_auc_gap_errorbars.png")
    plt.show()

def plot_dkt_utility_calibration():
    df = dkt_main_effect[dkt_main_effect["Attribute"] == CONFIG["PRIMARY_ATTR"]].sort_values("epsilon", ascending=False)
    x = np.arange(len(df))
    plt.figure(figsize=(7.2, 4.4))
    plt.errorbar(x, df["delta_auc_mean"], yerr=df["delta_auc_std"], marker="o", capsize=4, label="ΔAUC")
    plt.errorbar(x, df["delta_brier_mean"], yerr=df["delta_brier_std"], marker="o", capsize=4, label="ΔBrier")
    plt.axhline(0, linestyle="--", linewidth=1)
    plt.xticks(x, [f"ε={e:g}" for e in df["epsilon"]])
    plt.ylabel("Seed-matched delta")
    plt.title("DKT: privacy–utility and calibration trade-off")
    plt.legend()
    plt.grid(axis="y", alpha=0.3)
    savefig("fig2_dkt_utility_calibration.png")
    plt.show()

def plot_d9_robustness():
    df = d9_robustness.sort_values("epsilon", ascending=False)
    x = np.arange(len(df))
    labels = [f"ε={e:g}" for e in df["epsilon"]]
    width = 0.22
    plt.figure(figsize=(8.2, 4.6))
    plt.bar(x - width, df["bootstrap_delta_gap"], width, yerr=df["bootstrap_delta_gap_std"], capsize=3, label="Bootstrap")
    plt.bar(x, df["equal_size_delta_gap"], width, yerr=df["equal_size_delta_gap_std"], capsize=3, label="Equal-size")
    plt.bar(x + width, df["label_strat_delta_gap"], width, yerr=df["label_strat_delta_gap_std"], capsize=3, label="Label-stratified")
    plt.axhline(0, linestyle="--", linewidth=1)
    plt.xticks(x, labels)
    plt.ylabel("ΔAUC Gap")
    plt.title("Robustness: ranking gap persists after balancing")
    plt.legend()
    plt.grid(axis="y", alpha=0.3)
    savefig("fig3_d9_robustness_gap_checks.png")
    plt.show()

def plot_d10_mechanism():
    df = d10_mechanism.sort_values("epsilon", ascending=False)
    x = np.arange(len(df))
    labels = [f"ε={e:g}" for e in df["epsilon"]]
    width = 0.26
    plt.figure(figsize=(8.2, 4.6))
    plt.bar(x - width, df["loss_g1_minus_g0_mean"], width, yerr=df["loss_g1_minus_g0_std"], capsize=3, label="Loss g1-g0")
    plt.bar(x, df["grad_norm_g1_minus_g0_mean"], width, yerr=df["grad_norm_g1_minus_g0_std"], capsize=3, label="Grad norm g1-g0")
    plt.bar(x + width, df["clip_rate_g1_minus_g0_mean"], width, yerr=df["clip_rate_g1_minus_g0_std"], capsize=3, label="Clip rate g1-g0")
    plt.axhline(0, linestyle="--", linewidth=1)
    plt.xticks(x, labels)
    plt.ylabel("Group1 - group0")
    plt.title("Mechanism diagnostics: partial clipping evidence")
    plt.legend()
    plt.grid(axis="y", alpha=0.3)
    savefig("fig4_d10_mechanism_diagnostics.png")
    plt.show()

def plot_sakt_signed_disparity():
    df = sakt_robustness.sort_values(["Attribute", "epsilon"], ascending=[True, False])
    x = np.arange(len(df))
    plt.figure(figsize=(9.0, 4.6))
    plt.bar(x, df["delta_signed_mean"], yerr=df["delta_signed_std"], capsize=4)
    plt.axhline(0, linestyle="--", linewidth=1)
    plt.xticks(x, [f"{a}\nε={e:g}" for a, e in zip(df["Attribute"], df["epsilon"])], rotation=20, ha="right")
    plt.ylabel("Δ(AUC_lowIMD - AUC_reference)")
    plt.title("SAKT: signed low-IMD ranking degradation under DP-SGD")
    plt.grid(axis="y", alpha=0.3)
    savefig("fig5_sakt_signed_disparity.png")
    plt.show()

def lire_numeric_table():
    df = d15_lire_eps10.copy()
    for col in ["delta_auc", "delta_group1_auc", "delta_auc_gap", "delta_signed"]:
        means, stds = zip(*df[col].map(parse_pm))
        df[col + "_mean"] = means
        df[col + "_std"] = stds
    return df

def plot_lire_pareto_eps10():
    df = lire_numeric_table()
    show = df[df["Method"].isin(["WARM-GPR_lam0p02_w3", "LIRE-DP_a0p5", "LIRE-WGPR_a0p5_lam0p02_w3", "LIRE-DP_a1p0", "CLIP-C1p5"])].copy()

    plt.figure(figsize=(7.8, 5.2))
    for _, row in show.iterrows():
        plt.errorbar(
            row["delta_auc_mean"], 
            row["delta_signed_mean"], 
            xerr=row["delta_auc_std"], 
            yerr=row["delta_signed_std"], 
            marker="o", 
            capsize=3
        )
        plt.text(row["delta_auc_mean"] + 0.0007, row["delta_signed_mean"], row["Method"], fontsize=8)
    plt.axvline(0, linestyle="--", linewidth=1)
    plt.axhline(0, linestyle="--", linewidth=1)
    plt.xlabel("ΔAUC vs standard DP")
    plt.ylabel("ΔSigned disparity vs standard DP")
    plt.title("D15: LIRE improves ranking fairness with small utility cost")
    plt.grid(alpha=0.3)
    savefig("fig6_d15_lire_pareto_eps10.png")
    plt.show()

def plot_d16_confirmatory():
    df = d16_confirmatory.copy()
    x = np.arange(len(df))
    plt.figure(figsize=(9.0, 4.6))
    plt.bar(x, df["delta_signed_mean"], yerr=df["delta_signed_std"], capsize=4)
    plt.axhline(0, linestyle="--", linewidth=1)
    labels = [f"{m}\n{model}, ε={eps:g}" for m, model, eps in zip(df["Method"], df["Model"], df["epsilon"])]
    plt.xticks(x, labels, rotation=15, ha="right")
    plt.ylabel("ΔSigned disparity vs standard DP")
    plt.title("D16: LIRE confirms in DKT but not in SAKT")
    plt.grid(axis="y", alpha=0.3)
    savefig("fig7_d16_confirmatory_signed_disparity.png")
    plt.show()

def plot_lire_vs_clip_tradeoff():
    df = lire_numeric_table()
    show = df[df["Method"].isin(["LIRE-DP_a0p5", "LIRE-WGPR_a0p5_lam0p02_w3", "CLIP-C1p5"])].copy()
    x = np.arange(len(show))
    plt.figure(figsize=(7.8, 4.6))
    plt.bar(x - 0.18, show["delta_auc_mean"], 0.36, yerr=show["delta_auc_std"], capsize=4, label="ΔAUC")
    plt.bar(x + 0.18, show["delta_signed_mean"], 0.36, yerr=show["delta_signed_std"], capsize=4, label="ΔSigned disparity")
    plt.axhline(0, linestyle="--", linewidth=1)
    plt.xticks(x, show["Method"], rotation=18, ha="right")
    plt.ylabel("Seed-matched delta vs DP")
    plt.title("LIRE obtains similar fairness gain with far less utility loss than clipping sweep")
    plt.legend()
    plt.grid(axis="y", alpha=0.3)
    savefig("fig8_lire_vs_clip_tradeoff.png")
    plt.show()

# Generate all paper-ready figures.
plot_dkt_main_delta_gap()
plot_dkt_utility_calibration()
plot_d9_robustness()
plot_d10_mechanism()
plot_sakt_signed_disparity()
plot_lire_pareto_eps10()
plot_d16_confirmatory()
plot_lire_vs_clip_tradeoff()


In [ ]:

# ============================================================
# 13. Paper-ready table formatting and export
# ============================================================

def format_dkt_main(df):
    out = df.copy()
    out["ΔAUC"] = [pm_str(m, s) for m, s in zip(out["delta_auc_mean"], out["delta_auc_std"])]
    out["ΔBrier"] = [pm_str(m, s) for m, s in zip(out["delta_brier_mean"], out["delta_brier_std"])]
    out["Baseline AUC Gap"] = [pm_str(m, s) for m, s in zip(out["baseline_auc_gap_mean"], out["baseline_auc_gap_std"])]
    out["DP AUC Gap"] = [pm_str(m, s) for m, s in zip(out["dp_auc_gap_mean"], out["dp_auc_gap_std"])]
    out["ΔAUC Gap"] = [pm_str(m, s, signed=True) for m, s in zip(out["delta_auc_gap_mean"], out["delta_auc_gap_std"])]
    return out[["Model", "Attribute", "epsilon", "ΔAUC", "ΔBrier", "Baseline AUC Gap", "DP AUC Gap", "ΔAUC Gap", "interpretation"]]

def format_d9(df):
    out = df.copy()
    out["Raw ΔGap"] = [pm_str(m, s, signed=True) for m, s in zip(out["raw_delta_gap"], out["raw_delta_gap_std"])]
    out["Bootstrap ΔGap"] = [pm_str(m, s, signed=True) for m, s in zip(out["bootstrap_delta_gap"], out["bootstrap_delta_gap_std"])]
    out["Equal-size ΔGap"] = [pm_str(m, s, signed=True) for m, s in zip(out["equal_size_delta_gap"], out["equal_size_delta_gap_std"])]
    out["Label-stratified ΔGap"] = [pm_str(m, s, signed=True) for m, s in zip(out["label_strat_delta_gap"], out["label_strat_delta_gap_std"])]
    return out[["Attribute", "epsilon", "Raw ΔGap", "Bootstrap ΔGap", "Equal-size ΔGap", "Label-stratified ΔGap", "interpretation"]]

def format_d10(df):
    out = df.copy()
    out["Loss g1-g0"] = [pm_str(m, s) for m, s in zip(out["loss_g1_minus_g0_mean"], out["loss_g1_minus_g0_std"])]
    out["GradNorm g1-g0"] = [pm_str(m, s) for m, s in zip(out["grad_norm_g1_minus_g0_mean"], out["grad_norm_g1_minus_g0_std"])]
    out["Q95 GradNorm g1-g0"] = [pm_str(m, s) for m, s in zip(out["grad_norm_q95_g1_minus_g0_mean"], out["grad_norm_q95_g1_minus_g0_std"])]
    out["ClipRate g1-g0"] = [pm_str(m, s) for m, s in zip(out["clip_rate_g1_minus_g0_mean"], out["clip_rate_g1_minus_g0_std"])]
    return out[["Attribute", "epsilon", "Loss g1-g0", "GradNorm g1-g0", "Q95 GradNorm g1-g0", "ClipRate g1-g0", "interpretation"]]

def format_sakt(df):
    out = df.copy()
    out["ΔAUC"] = [pm_str(m, s) for m, s in zip(out["delta_auc_mean"], out["delta_auc_std"])]
    out["ΔBrier"] = [pm_str(m, s) for m, s in zip(out["delta_brier_mean"], out["delta_brier_std"])]
    out["ΔGroup1 AUC"] = [pm_str(m, s) for m, s in zip(out["delta_group1_auc_mean"], out["delta_group1_auc_std"])]
    out["Baseline Gap"] = [pm_str(m, s) for m, s in zip(out["baseline_auc_gap_mean"], out["baseline_auc_gap_std"])]
    out["DP Gap"] = [pm_str(m, s) for m, s in zip(out["dp_auc_gap_mean"], out["dp_auc_gap_std"])]
    out["ΔAUC Gap"] = [pm_str(m, s, signed=True) for m, s in zip(out["delta_auc_gap_mean"], out["delta_auc_gap_std"])]
    out["ΔSigned"] = [pm_str(m, s, signed=True) for m, s in zip(out["delta_signed_mean"], out["delta_signed_std"])]
    return out[["Model", "Attribute", "epsilon", "ΔAUC", "ΔBrier", "ΔGroup1 AUC", "Baseline Gap", "DP Gap", "ΔAUC Gap", "ΔSigned", "interpretation"]]

def format_d16(df):
    out = df.copy()
    out["ΔAUC"] = [pm_str(m, s) for m, s in zip(out["delta_auc_mean"], out["delta_auc_std"])]
    out["ΔBrier"] = [pm_str(m, s) for m, s in zip(out["delta_brier_mean"], out["delta_brier_std"])]
    out["ΔGroup1 AUC"] = [pm_str(m, s) for m, s in zip(out["delta_group1_auc_mean"], out["delta_group1_auc_std"])]
    out["ΔAUC Gap"] = [pm_str(m, s, signed=True) for m, s in zip(out["delta_auc_gap_mean"], out["delta_auc_gap_std"])]
    out["ΔSigned"] = [pm_str(m, s, signed=True) for m, s in zip(out["delta_signed_mean"], out["delta_signed_std"])]
    return out[["Stage", "Model", "Method", "Attribute", "epsilon", "ΔAUC", "ΔBrier", "ΔGroup1 AUC", "ΔAUC Gap", "ΔSigned", "interpretation"]]

paper_tables = {
    "table1_dkt_main_effect": format_dkt_main(dkt_main_effect),
    "table2_metric_dependency": metric_dependency_table,
    "table3_d9_robustness": format_d9(d9_robustness),
    "table4_d10_mechanism": format_d10(d10_mechanism),
    "table5_sakt_robustness": format_sakt(sakt_robustness),
    "table6_d11_static_weighting": d11_static_weighting,
    "table7_d13_pairrank_clipsweep": d13_pairrank_clipsweep,
    "table8_d14_clipaware": d14_clipaware,
    "table9_d15_lire_eps10": d15_lire_eps10,
    "table10_d15_lire_diagnostics": d15_lire_diagnostics,
    "table11_d16_confirmatory": format_d16(d16_confirmatory),
    "table12_d16_diagnostics": d16_diagnostics,
}

for name, df in paper_tables.items():
    csv_path = RESULT_DIR / f"{name}.csv"
    tex_path = RESULT_DIR / f"{name}.tex"
    df.to_csv(csv_path, index=False)
    try:
        df.to_latex(tex_path, index=False, escape=False)
    except Exception as e:
        print(f"Skipping LaTeX export for {name}: {e}")
    print(f"Saved {name}: {csv_path}")
    display(df)

# A compact final table for the manuscript main text.
# Avoid pandas.query here: some Kaggle/Pandas/NumExpr versions fail when query
# expressions reference nested objects such as CONFIG['PRIMARY_ATTR'].
primary_attr = CONFIG["PRIMARY_ATTR"]
mitigation_methods = ["LIRE-DP_a0p5", "LIRE-WGPR_a0p5_lam0p02_w3", "CLIP-C1p5"]

main_text_tables = {
    "main_table_dkt": paper_tables["table1_dkt_main_effect"].loc[
        paper_tables["table1_dkt_main_effect"]["Attribute"].eq(primary_attr)
    ].copy(),
    "main_table_mitigation": paper_tables["table9_d15_lire_eps10"].loc[
        paper_tables["table9_d15_lire_eps10"]["Method"].isin(mitigation_methods)
    ].copy(),
    "main_table_confirmation": paper_tables["table11_d16_confirmatory"].copy(),
}

for name, df in main_text_tables.items():
    csv_path = RESULT_DIR / f"{name}.csv"
    tex_path = RESULT_DIR / f"{name}.tex"
    df.to_csv(csv_path, index=False)
    try:
        df.to_latex(tex_path, index=False, escape=False)
    except Exception as e:
        print(f"Skipping LaTeX export for {name}: {e}")
    print(f"Saved {name}: {csv_path}")
    display(df)


# Interpretation guide for the Results section

## RQ1: Does DP-SGD introduce utility and calibration degradation in KT?

Yes. DKT shows large AUC degradation as ε decreases, with positive ΔBrier values. SAKT independently confirms strong AUC and Brier degradation. This supports the conclusion that the privacy–utility and calibration trade-off is not DKT-specific.

## RQ2: Is the fairness effect metric-dependent?

Yes. The strongest and most interpretable effect is ranking-based: AUC Gap and signed AUC disparity. DP Gap and EO/Opportunity-style threshold-rate metrics do not show the same consistent deterioration pattern. The paper should therefore argue **metric-dependent fairness effects**, not uniform fairness harm across all definitions.

## RQ3: Is the IMD ranking disparity only a group-size or label-imbalance artifact?

No. D9 robustness checks show that bootstrap, equal-size resampling, and label-stratified balancing retain the positive ΔAUC Gap for ε=10 and ε=1. ε=0.5 is positive but less stable because the model is closer to severe utility degradation.

## RQ4: What mechanism is supported?

D10 supports a **partial clipping-distortion mechanism**. Under ε=10, low-IMD samples have higher loss, higher gradient norms, and higher clipping rates. Under ε=1, clipping evidence is weak, so clipping cannot be the only explanation.

## RQ5: Do simple mitigation methods work?

Ordinary static reweighting (D11) and simple PairRank/CAPR variants (D13/D14) produce limited or negligible improvements. Clipping-threshold sweep strongly reduces ranking disparity but causes large AUC/Brier degradation, making it mechanism evidence rather than a practical mitigation.

## RQ6: What is the best mitigation candidate?

LIRE-WGPR is the best DKT-specific candidate. At ε=10, it improves low-IMD AUC and reduces AUC Gap with much smaller utility cost than CLIP-C1.5. D16 confirms the effect under ε=1 for DKT. However, SAKT-LIRE does not reduce low-IMD ranking disparity, so the mitigation is currently **architecture-dependent**.

## Final claim boundary

The final manuscript should say:

> DP-SGD induces metric-dependent fairness effects in educational knowledge tracing. Ranking-based IMD disparity is the clearest fairness risk. Clipping distortion partially explains the effect. LIRE-WGPR provides a promising DKT-specific mitigation, but architecture-general mitigation remains open.

Avoid saying:

> LIRE solves DP fairness risk in KT.

A more accurate statement is:

> LIRE-WGPR is the first practically useful DKT-specific mitigation candidate in this study, but DP-induced ranking fairness risk remains only partially solved.


# Recommended paper narrative

A compact Results/Discussion structure:

1. **Dataset and sensitive attribute.** Use OULAD because it provides `imd_band`; define `imd_low30_vs_rest` as the main socioeconomic disadvantage contrast and `imd_low30_vs_high70` as a cleaner high-vs-low robustness contrast.
2. **Main effect in DKT.** DP-SGD causes strong AUC/Brier degradation and expands IMD ranking disparity.
3. **Metric dependency.** Ranking metrics show the clearest fairness risk; DP/EO-style threshold metrics do not deteriorate uniformly.
4. **Robustness to imbalance.** Bootstrap, equal-size, and label-stratified checks rule out trivial group-size/label-distribution explanations.
5. **Mechanism.** Clipping distortion is partly supported, especially under ε=10, but does not fully explain all privacy settings.
6. **Architecture robustness.** SAKT confirms privacy–utility degradation and signed low-IMD degradation; high-vs-low IMD AUC Gap also enlarges.
7. **Mitigation probes.** Static weighting, PairRank, and CAPR are weak; clipping-threshold sweep is useful mechanism evidence but not practical.
8. **LIRE.** LIRE-WGPR provides the best DKT-specific Pareto mitigation and is confirmed under ε=1, but does not transfer cleanly to SAKT.
9. **Limitations.** Single dataset with socioeconomic labels, architecture-dependent mitigation, and no formal proof for LIRE's optimality or universal DP-fairness guarantee.
10. **Future work.** Group-wise private optimization, clipping-bias correction, DP-aware robust objectives, and architecture-specific residual experts.


# Bibliography anchors

Use the venue's required citation style in the manuscript. These are the references this notebook expects the paper to cite.

1. Piech, C., Spencer, J., Huang, J., Ganguli, S., Sahami, M., Guibas, L., & Sohl-Dickstein, J. (2015). **Deep Knowledge Tracing**. *NeurIPS*.
2. Pandey, S., & Karypis, G. (2019). **A Self-Attentive Model for Knowledge Tracing**. *EDM*.
3. Ghosh, A., Heffernan, N., & Lan, A. S. (2020). **Context-Aware Attentive Knowledge Tracing**. *KDD*.
4. Kuzilek, J., Hlosta, M., & Zdrahal, Z. (2017). **Open University Learning Analytics Dataset**. *Scientific Data*.
5. Abadi, M. et al. (2016). **Deep Learning with Differential Privacy**. *CCS*.
6. Opacus documentation. **PrivacyEngine and privacy accounting API**.
7. Bagdasaryan, E., & Shmatikov, V. (2019). **Differential Privacy Has Disparate Impact on Model Accuracy**.
8. Mangold, P., Bellet, A., Tommasi, M., & Vincent, E. (2023). **Differential Privacy has Bounded Impact on Fairness in Classification**. *ICML*.
9. Hansen, N. et al. (2024). **Does Differential Privacy Really Protect Underrepresented Groups?** *Findings of NAACL*.
10. Recent work on clipping bias, adaptive clipping, and error-feedback DP-SGD can be cited in the mitigation/future-work section, especially when discussing why clipping distortion may affect group fairness.

**Citation discipline.** Claims about OULAD fields, DKT/SAKT/AKT architecture provenance, and Opacus privacy accounting should be cited in the paper. Claims about LIRE are this work's contribution and should be clearly marked as proposed in this paper.
